# 04 Assumption Checks

Shapiro-Wilk検定とLevene検定を連続で実行し、CSVに保存するNotebook。

- 入力: `data/processed/{DATASET_NAME}/team_centroids_by_sequence.csv`
- 出力: `outputs/tables/{DATASET_NAME}/assumption_checks.csv`
- 有意水準: `alpha = 0.05`
- nが3未満のグループはShapiro-Wilk検定を実行しない
- nが20未満のグループは `note` に注意を出す

In [ ]:
from pathlib import Path

import pandas as pd
from scipy.stats import levene, shapiro

def find_project_root(start=None):
    current = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'data').exists() and (candidate / 'はるた').exists():
            return candidate
    raise FileNotFoundError('プロジェクトルートが見つかりません。data/ と はるた/ がある場所で実行してください。')


PROJECT_ROOT = find_project_root()
HARUTA_ROOT = PROJECT_ROOT / 'はるた'
MATCH_DATE = '2023-11-25'
MATCH_ID = '117093'
MATCH_LABEL = 'tsukuba_vs_tsukuba-b'
DATASET_NAME = f'{MATCH_DATE}_{MATCH_ID}_{MATCH_LABEL}'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed' / DATASET_NAME
TABLES_DIR = HARUTA_ROOT / 'outputs' / 'tables' / DATASET_NAME
TABLES_DIR.mkdir(parents=True, exist_ok=True)
INPUT_PATH = PROCESSED_DIR / 'team_centroids_by_sequence.csv'
OUTPUT_PATH = TABLES_DIR / 'assumption_checks.csv'
ALPHA = 0.05
MIN_SHAPIRO_N = 3
SHAPIRO_WARNING_N = 20
REQUIRED_COLUMNS = ['group', 'mean_centroid_x', 'mean_centroid_y']
VALUE_COLS = ['mean_centroid_x', 'mean_centroid_y']


In [ ]:
def load_centroid_sequences(path):
    if not path.exists():
        raise FileNotFoundError(f'重心分析用CSVが見つかりません: {path}')
    df = pd.read_csv(path)
    missing = [col for col in REQUIRED_COLUMNS if col not in df.columns]
    if missing:
        raise ValueError(f'必須列が不足しています: {missing}')
    if df.empty:
        raise ValueError('入力CSVが0行です。先に 00_data_check.ipynb を再実行してください。')
    return df


def shapiro_wilk_by_group(df, value_col, group_col='group', alpha=ALPHA):
    rows = []
    for group, sub in df.groupby(group_col):
        values = sub[value_col].dropna().to_numpy()
        n = len(values)
        checked = n >= MIN_SHAPIRO_N
        if not checked:
            statistic, p_value = float('nan'), float('nan')
            note = f'n={n} のためShapiro-Wilk検定は実行不可。最低 {MIN_SHAPIRO_N} 件必要。'
        else:
            statistic, p_value = shapiro(values)
            note = 'nが少ないため解釈注意。' if n < SHAPIRO_WARNING_N else ''
        rows.append({
            'test': 'shapiro_wilk',
            'variable': value_col,
            'group': group,
            'n': n,
            'statistic': statistic,
            'p_value': p_value,
            'alpha': alpha,
            'passed': bool(pd.notna(p_value) and p_value > alpha),
            'checked': checked,
            'note': note,
        })
    return pd.DataFrame(rows)


def levene_across_groups(df, value_col, group_col='group', alpha=ALPHA):
    samples = [sub[value_col].dropna().to_numpy() for _, sub in df.groupby(group_col)]
    samples = [sample for sample in samples if len(sample) > 0]
    if len(samples) < 2:
        statistic, p_value = float('nan'), float('nan')
        checked = False
        note = 'Levene検定には2グループ以上必要です。00_data_check.ipynb の出力を確認してください。'
    else:
        statistic, p_value = levene(*samples)
        checked = True
        note = ''
    return pd.DataFrame([{
        'test': 'levene',
        'variable': value_col,
        'group': 'all',
        'n': sum(len(sample) for sample in samples),
        'statistic': statistic,
        'p_value': p_value,
        'alpha': alpha,
        'passed': bool(pd.notna(p_value) and p_value > alpha),
        'checked': checked,
        'note': note,
    }])


def run_assumption_tests(df, value_cols=VALUE_COLS):
    parts = []
    for col in value_cols:
        parts.append(shapiro_wilk_by_group(df, col))
        parts.append(levene_across_groups(df, col))
    return pd.concat(parts, ignore_index=True)

In [ ]:
df = load_centroid_sequences(INPUT_PATH)
df.head()

In [ ]:
results = run_assumption_tests(df)
results

In [ ]:
results.to_csv(OUTPUT_PATH, index=False)
print(f'Saved: {OUTPUT_PATH}')